In [1]:
import geopandas as gpd
import numpy as np
import math
import random as rnd
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from svg_plot import SVGPlot
import time
import csv

In [2]:
class GeometryUtils:
    """Static utility methods for geometric operations."""
    
    @staticmethod
    def point_in_polygon(x: float, y: float, polygon: List[Tuple[float, float]]) -> bool:
        """Ray casting algorithm to check if point is inside polygon."""
        polygon = np.asarray(polygon)
        n = len(polygon)
        inside = False

        for i in range(n):
            x1, y1 = polygon[i]
            x2, y2 = polygon[(i + 1) % n]

            # Check if edge crosses horizontal ray to the right
            if ((y1 > y) != (y2 > y)) and (x < (x2 - x1) * (y - y1) / (y2 - y1) + x1):
                inside = not inside

        return inside

    @staticmethod
    def get_bounding_box(vertices: List[Tuple[float, float]]) -> Tuple[float, float, float, float]:
        """Returns (x_min, x_max, y_min, y_max)."""
        x_vals = [v[0] for v in vertices]
        y_vals = [v[1] for v in vertices]
        return min(x_vals), max(x_vals), min(y_vals), max(y_vals)


In [3]:
@dataclass
class MigrationStrategy:
    """Encapsulates migration probability logic."""
    
    @staticmethod
    def linear_toward_zero() -> float:
        """Generates a value biased towards zero (both positive and negative)."""
        u = rnd.random()
        magnitude = 1 - math.sqrt(1 - u)
        
        # Randomly assign sign (+ or -)
        sign = 1 if rnd.random() < 0.5 else -1
        
        return sign * magnitude

    @staticmethod
    def age_migrate_curve(x: float, start_point: int = 20, end_point: int = 70) -> float:
        """Calculates migration probability based on age."""
        if x < start_point:
            return 0.0
        
        # Decay rate calculation
        decay_rate = -np.log(0.01) / (end_point - start_point)
        shifted_x = x - start_point
        prob = np.exp(-decay_rate * shifted_x)
        return float(np.clip(prob, 0, 1))



In [4]:
class PopulationArea:
    """Represents a geographic administrative area."""
    
    def __init__(self, area_id: str, admin_area: str, area_type: str, vertices: List[Tuple[float, float]]):
        self.area_id = area_id
        self.admin_area = admin_area
        self.area_type = area_type
        self.vertices = vertices
        self.local_population: List['Denizen'] = []
        
        # Pre-calculate bounding box for faster random generation
        self.x_min, self.x_max, self.y_min, self.y_max = GeometryUtils.get_bounding_box(vertices)

    def add_person(self, person: 'Denizen') -> None:
        self.local_population.append(person)

    def remove_person(self, person: 'Denizen') -> None:
        if person in self.local_population:
            self.local_population.remove(person)

    def find_containing_area(self, x: float, y: float) -> bool:
        return GeometryUtils.point_in_polygon(x, y, self.vertices)


In [14]:
class Denizen:
    """Represents an individual agent in the simulation."""
    def __init__(self, initial_area: PopulationArea,row):
        self.area = initial_area
        self.pos: Tuple[float, float] = (0.0, 0.0)
        self.age = int(row['age'])
        self.moved = False
 
        
    def assign_position(self) -> None:
        """Assigns a random valid position within the current area."""
        while True:
            x = self.area.x_min + rnd.random() * (self.area.x_max - self.area.x_min)
            y = self.area.y_min + rnd.random() * (self.area.y_max - self.area.y_min)

            if self.area.find_containing_area(x, y):
                self.pos = (float(x), float(y))
                self.area.add_person(self)
                return

    def should_migrate(self) -> bool:
        """Determines if the denizen attempts to migrate based on age."""
        return rnd.random() < MigrationStrategy.age_migrate_curve(self.age)

    def migrate(self, all_areas: Dict[str, PopulationArea]) -> bool:
        """
        Attempts to migrate to a new area.
        Returns True if migration was successful.
        """
        if self.moved:
            return False
            
        if not self.should_migrate():
            return False

        # Attempt migration
        for _ in range(100): # Limit attempts to prevent infinite loops
            x,y = self.pos
            x += MigrationStrategy.linear_toward_zero() * 10
            y += MigrationStrategy.linear_toward_zero() * 10
            
            # Find new area
            new_area = None
            for area in all_areas.values():
                if area.find_containing_area(x, y):
                    new_area = area
                    break
            
            if new_area and new_area.area_id != "z": # Exclusion logic from original
                # Update state
                self.area.remove_person(self)
                self.area = new_area
                self.pos = (float(x), float(y))
                self.area.add_person(self)
                self.moved = True
                return True
                
        return False

    def __str__(self) -> str:
        return f"Denizen(ID={self.area.area_id}, Pos={self.pos}, Age={self.age})"

In [15]:
class MigrationSimulation:
    """Manages the entire simulation lifecycle, data loading, and stepping."""
    
    def __init__(self, geojson_path: str, csv_path: str, scale_factor: float = 1.0):
        self.areas: Dict[str, PopulationArea] = {}
        self.population: List[Denizen] = []
        self.strategy = MigrationStrategy()
        
        self._load_geojson(geojson_path, scale_factor)
        self._load_population(csv_path)

    def _load_geojson(self, file_path: str, scale_factor: float) -> None:
        """Loads areas from GeoJSON file."""
        gdf = gpd.read_file(file_path)
        
        for row in gdf.itertuples(index=False):
            geom = row.geometry
            if geom is None:
                continue

            # Handle Polygon geometry
            if geom.geom_type == 'Polygon':
                coords = [
                    [pt[0] * scale_factor, pt[1] * scale_factor] 
                    for pt in geom.exterior.coords
                ]
                
                # Extract properties, skipping geometry
                props = {k: v for k, v in row._asdict().items() if k != 'geometry'}
                
                area_id = str(props.get('id', ''))
                if area_id:
                    area = PopulationArea(
                        area_id=area_id,
                        admin_area=props.get('admin_area', ''),
                        area_type=props.get('type', ''),
                        vertices=coords
                    )
                    self.areas[area_id] = area

    def _load_population(self, csv_path: str) -> None:
        """Loads initial population from CSV and assigns positions."""
        with open(csv_path, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                area_id = row['area']
                if area_id in self.areas:
                    person = Denizen(self.areas[area_id], row)
                    person.assign_position()
                    self.population.append(person)
                else:
                    print(f"Warning: Area ID '{area_id}' not found in GeoJSON. Skipping person.")

    def step(self) -> None:
        """Executes one iteration of the simulation."""
        for person in self.population:
            person.migrate(self.areas)

    def get_state(self) -> Tuple[List[Denizen], Dict[str, PopulationArea]]:
        """Returns current state for visualization."""
        return self.population, self.areas

In [20]:
def run_visualization(sim: MigrationSimulation, iterations: int = 100):
    """Runs the simulation loop with SVG visualization."""
    if SVGPlot is None:
        print("Visualization skipped: SVGPlot module not available.")
        return

    svg = SVGPlot(500, 500, 12)
    
    # Initial display setup
    from IPython.display import display, HTML
    output_display = display(HTML("<div id='sim-container'>Loading...</div>"), display_id=True)

    for t in range(iterations):
        svg.clear()
        
        # Draw Areas
        for area in sim.areas.values():
            svg.add_polygon(area.vertices, 'rgb(220,220,220)')
        
        # Draw People
        for p in sim.population:
            if p.moved:
                x, y = p.pos
                # Ensure RGB values are within 0-255
                r = 255
                g = int(min(255, p.age * 3))
                b = int(min(255, p.age * 3))
                svg.add_rect(x, y, f'rgb({r},{g},{b}', 0.05)
        
        # Run next step
        sim.step()
        
        new_svg = svg.get_canvas()
        new_svg += f'<p>Iteration: {t} weeks</p>'
        output_display.update(HTML(f"<div id='sim-container'>{new_svg}</div>"))
        time.sleep(0.01)

In [21]:

sim = MigrationSimulation(
    geojson_path='Meropis/bounded_polygons_small.geojson',
    csv_path='Meropis/artifical_population.csv',
    scale_factor=10.0
)

print(f"Loaded {len(sim.areas)} areas and {len(sim.population)} people.")

# Run Visualization
run_visualization(sim, iterations=100)
    


Loaded 11 areas and 5891 people.
